# ARCHIVED — early probe of the frozen detector

The notebook that first established the generate-then-detect loop works: the detector never
fired at the lesion site on an untouched chest and always fired once a lesion was painted.

**Superseded.** That result (recorded as P1) was measured on one chest and does not
generalise as stated — the 12-chest grid gives a mean of 0.815 with genuine failures, not
20 out of 20. Detection also turned out to be insensitive to input resolution, since the
detector's internal transform rescales everything to ~800 px regardless.

Kept as provenance for how the resolution question was settled.

In [ ]:
from pathlib import Path
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision
from PIL import Image
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as TF

SCORE_MIN  = 0.05
SIZES      = [512, 800, 1024, 1536]
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
print(DEVICE)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT = '/content/drive/MyDrive/Algoverse/Teammates/baseline1_checkpoint.pth'   # the frozen detector

In [ ]:
def load_detector(path, score_thresh=0.0, detections=300):
    m = torchvision.models.detection.fasterrcnn_resnet50_fpn(
        weights=None, weights_backbone=None,
        box_score_thresh=score_thresh, box_detections_per_img=detections)
    feats = m.roi_heads.box_predictor.cls_score.in_features
    m.roi_heads.box_predictor = FastRCNNPredictor(feats, 2)

    state = torch.load(path, map_location=DEVICE)
    m.load_state_dict(state['model'] if 'model' in state else state)
    m.eval().to(DEVICE)

    t = m.transform
    print(f'internal transform: min_size={t.min_size}, max_size={t.max_size}')
    print('  -> inputs are rescaled to that range before the backbone sees them')
    return m

model = load_detector(CHECKPOINT)

In [ ]:
@torch.no_grad()
def detect(model, pil, size):
    """Returns boxes and scores as FRACTIONS of the image."""
    img = pil.convert('RGB').resize((size, size), Image.LANCZOS)
    out = model([TF.to_tensor(img).to(DEVICE)])[0]
    return out['boxes'].cpu().numpy() / size, out['scores'].cpu().numpy()


def centre_in_box(pred, target):
    cx, cy = (pred[0] + pred[2]) / 2, (pred[1] + pred[3]) / 2
    return target[0] <= cx <= target[2] and target[1] <= cy <= target[3]


def summarise(label, boxes, scores, target):
    keep = scores >= SCORE_MIN
    b, s = boxes[keep], scores[keep]
    hits = [ss for bb, ss in zip(b, s) if centre_in_box(bb, target)]
    best = max(hits, default=0.0)
    print(f'  {label:<24} boxes: {len(b):>3}   on lesion: {len(hits):>2}   '
          f'best on lesion: {best:.3f}   top anywhere: {s.max() if len(s) else 0:.3f}')
    return len(b), len(hits), best

In [ ]:
from pathlib import Path
from PIL import Image

IMAGE_DIR = Path('/content/drive/MyDrive/Algoverse/Me/radedit_qc')

canvas = Image.open(IMAGE_DIR / 'canvas.png').convert('RGB')
edits  = [Image.open(IMAGE_DIR / f'testing{i}.png').convert('RGB') for i in range(5)]

print(f'canvas {canvas.size}, {len(edits)} edits, first {edits[0].size}')
display(canvas, edits[0])

# The nominal box was 5% of the image at (0.32, 0.40). But the mask was that box plus
# 24px padding per side = 74px = 14.5% of the image, and the calibration showed RadEdit
# fills the whole mask rather than painting a small nodule inside it. So the mask extent
# is where the change actually is.

NOMINAL = (0.32, 0.40, 0.05,  0.05)
MASK    = (0.32, 0.40, 0.145, 0.145)

TESTS = [(f'seed{i}', canvas, edits[i], MASK) for i in range(5)]
print(f'{len(TESTS)} pairs, target = mask extent')

In [ ]:
import glob
for p in glob.glob('/content/drive/MyDrive/**/*.png', recursive=True)[:40]:
    print(p)

In [ ]:
rows = []
for name, bg, edited, (cx, cy, w, h) in TESTS:
    target = (cx - w/2, cy - h/2, cx + w/2, cy + h/2)
    print(f'\n=== {name} ===  lesion {target[0]:.3f},{target[1]:.3f} '
          f'-> {target[2]:.3f},{target[3]:.3f}')
    for size in SIZES:
        print(f' input {size}px')
        nb, hb, best_bg = summarise('BACKGROUND (control)', *detect(model, bg, size), target)
        ne, he, best_ed = summarise('EDITED',               *detect(model, edited, size), target)
        rows.append({'case': name, 'input_px': size,
                     'bg_on_lesion': hb, 'bg_best': round(best_bg, 3),
                     'ed_on_lesion': he, 'ed_best': round(best_ed, 3),
                     'gain': round(best_ed - best_bg, 3)})

df = pd.DataFrame(rows)
print('\n', df.to_string(index=False))

In [ ]:
if df.empty:
    print('no tests defined')
elif (df.ed_on_lesion == 0).all():
    print('The detector never fires on the lesion. Generation quality or resolution is')
    print('the problem, and nothing downstream fixes it.')
elif (df.gain <= 0.05).all():
    print('Edited scores no better than the untouched background at the same spot.')
    print('Whatever it is firing on, it is not the lesion.')
else:
    b = df.loc[df.gain.idxmax()]
    print(f'Best separation at {b.input_px}px: edited {b.ed_best:.3f} vs background {b.bg_best:.3f}')
    if b.input_px != 512:
        print(f'-> upscaling to {b.input_px} helps. Feed the detector larger inputs even')
        print('   though RadEdit generates at 512.')

In [ ]:
def show(model, bg, edited, target, size=512, top=6):
    fig, axes = plt.subplots(1, 2, figsize=(11, 5.6))
    for ax, (img, title) in zip(axes, [(bg, 'background'), (edited, 'edited')]):
        boxes, scores = detect(model, img, size)
        keep = scores >= SCORE_MIN
        boxes, scores = boxes[keep], scores[keep]
        ax.imshow(img.convert('L'), cmap='gray', extent=[0, 1, 1, 0])
        ax.add_patch(patches.Rectangle((target[0], target[1]),
                                       target[2]-target[0], target[3]-target[1],
                                       fill=False, edgecolor='lime', lw=2))
        for i in np.argsort(-scores)[:top]:
            b, s = boxes[i], scores[i]
            on = centre_in_box(b, target)
            ax.add_patch(patches.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1],
                         fill=False, edgecolor='red' if on else 'orange', lw=1.5))
            ax.text(b[0], b[1]-0.01, f'{s:.2f}',
                    color='red' if on else 'orange', fontsize=7)
        ax.set_title(f'{title} — {len(boxes)} boxes ≥ {SCORE_MIN}', fontsize=10)
        ax.set_xlim(0, 1); ax.set_ylim(1, 0); ax.axis('off')
    plt.suptitle('green = lesion   red = box centred on it   orange = elsewhere')
    plt.tight_layout(); plt.show()

name, bg, edited, (cx, cy, w, h) = TESTS[0]
show(model, bg, edited, (cx-w/2, cy-h/2, cx+w/2, cy+h/2))

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Automatically retrieves the token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')
if hf_token:
    login(hf_token)
    print("Successfully logged in to Hugging Face!")
else:
    print("HF_TOKEN secret not found. Check your Colab sidebar.")

In [ ]:
from transformers import AutoModel, AutoTokenizer
from diffusers import (AutoencoderKL, DDIMScheduler, StableDiffusionPipeline,
                       UNet2DConditionModel, DiffusionPipeline)

unet  = UNet2DConditionModel.from_pretrained("microsoft/radedit", subfolder="unet")
vae   = AutoencoderKL.from_pretrained("stabilityai/sdxl-vae")
te    = AutoModel.from_pretrained("microsoft/BiomedVLP-BioViL-T", trust_remote_code=True)
tok   = AutoTokenizer.from_pretrained("microsoft/BiomedVLP-BioViL-T",
                                      model_max_length=128, trust_remote_code=True)
sched = DDIMScheduler(beta_schedule="linear", clip_sample=False, prediction_type="epsilon",
                      timestep_spacing="trailing", steps_offset=1)

gen = StableDiffusionPipeline(vae=vae, text_encoder=te, tokenizer=tok, unet=unet,
                              scheduler=sched, safety_checker=None,
                              requires_safety_checker=False, feature_extractor=None).to(DEVICE)
edit = DiffusionPipeline.from_pipe(gen, custom_pipeline="microsoft/radedit",
                                   trust_remote_code=True)
print('radedit ready')

In [ ]:
from PIL import ImageDraw, ImageOps

SIZE      = 512     # RadEdit working size
DET_SIZE  = 800     # yesterday's best -- reuse `detect` at this size
EDIT_MIN  = 12.0    # from the calibration; below this nothing was painted

BASE = dict(cx=0.32, cy=0.40, box=0.05, pad=24,
            skip=0.3, guidance=7.5, steps=200,
            prompt='Right upper lobe pulmonary nodule')

def make_mask(cx, cy, box, pad, size=SIZE):
    half = box * size / 2
    x0, y0 = cx*size - half - pad, cy*size - half - pad
    x1, y1 = cx*size + half + pad, cy*size + half + pad
    assert 0 <= x0 < x1 <= size and 0 <= y0 < y1 <= size, 'mask out of bounds'
    m = Image.new('L', (size, size), 0)
    ImageDraw.Draw(m).ellipse([x0, y0, x1, y1], fill=255)
    return m, (x0/size, y0/size, x1/size, y1/size), round(x1-x0, 1)

def edit_inside(before, after, mask):
    b = np.asarray(before.convert('L'), np.float32)
    a = np.asarray(after.convert('L'),  np.float32)
    m = np.asarray(mask, bool)
    d = np.abs(a - b)
    return float(d[m].mean()), float(d[~m].mean())

def best_at(pil, target, size=DET_SIZE):
    """Highest score whose box centre lands in `target`. Reuses detect + centre_in_box."""
    boxes, scores = detect(model, pil, size)
    keep = scores >= SCORE_MIN
    hits = [s for b, s in zip(boxes[keep], scores[keep]) if centre_in_box(b, target)]
    return float(max(hits, default=0.0))

_bg_cache = {}
def bg_score(target):
    key = tuple(round(t, 4) for t in target)
    if key not in _bg_cache:
        _bg_cache[key] = best_at(canvas, target)
    return _bg_cache[key]

def run_one(label, **kw):
    p = {**BASE, **kw}
    mask, target, mask_px = make_mask(p['cx'], p['cy'], p['box'], p['pad'])
    torch.manual_seed(42)
    out = edit(p['prompt'], weights=[p['guidance']], image=canvas, edit_mask=mask,
               keep_mask=ImageOps.invert(mask), num_inference_steps=p['steps'],
               invert_prompt='', skip_ratio=p['skip'], output_type='pil')[0]
    inside, outside = edit_inside(canvas, out, mask)
    ed, bg = best_at(out, target), bg_score(target)
    row = dict(case=label, mask_px=mask_px, skip=p['skip'], guidance=p['guidance'],
               pos=f"{p['cx']:.2f},{p['cy']:.2f}", prompt=p['prompt'][:34],
               edit_inside=round(inside, 2), painted=inside >= EDIT_MIN,
               det_edited=round(ed, 3), det_bg=round(bg, 3), gain=round(ed-bg, 3))
    print(f"{label:<22} mask={mask_px:>5}  edit={inside:6.2f}  det={ed:.3f}  "
          f"bg={bg:.3f}  gain={ed-bg:+.3f}"
          f"{'' if inside >= EDIT_MIN else '   <-- NOTHING PAINTED'}")
    return row, out

In [ ]:
SWEEP = [
    ('baseline',         {}),
    ('mask 47px',        dict(box=0.03, pad=16)),
    ('skip 0.5',         dict(skip=0.5)),
    ('skip 0.7',         dict(skip=0.7)),
    ('guidance 3.0',     dict(guidance=3.0)),
    ('prompt subtle',    dict(prompt='Subtle ill-defined pulmonary opacity')),
    ('pos mediastinal',  dict(cx=0.43)),
    ('pos apex',         dict(cy=0.22)),
    ('pos retrocardiac', dict(cx=0.40, cy=0.60)),
]

sweep_rows, sweep_imgs = [], {}
for label, kw in SWEEP:
    r, img = run_one(label, **kw)
    sweep_rows.append(r); sweep_imgs[label] = img

sweep = pd.DataFrame(sweep_rows)
print('\n', sweep.to_string(index=False))

In [ ]:
ok   = sweep[sweep.painted]
band = ok[(ok.det_edited >= 0.1) & (ok.det_edited <= 0.5) & (ok.gain > 0.05)]

print(f'{len(ok)}/{len(sweep)} actually painted something')
print(f'baseline detector score: {sweep.loc[sweep.case=="baseline","det_edited"].iloc[0]:.3f}\n')

if len(band):
    print('IN THE 0.1-0.5 BAND -- Route 3 is alive:')
    print(band[['case','mask_px','skip','guidance','pos','edit_inside','det_edited','gain']]
          .to_string(index=False))
else:
    print('Nothing landed in 0.1-0.5 with a real edit.')
    print('Knobs ranked by how much they lowered the score:')
    print(ok.sort_values('det_edited')[['case','edit_inside','det_edited','gain']]
            .to_string(index=False))

if len(sweep[~sweep.painted]):
    print(f'\nNOT painted, scores meaningless: {list(sweep[~sweep.painted].case)}')

In [ ]:
# edit these based on Cell 12
COMBOS = [
    ('subtle + small',    dict(box=0.03, pad=16, prompt='Subtle ill-defined pulmonary opacity')),
    ('subtle + skip 0.5', dict(skip=0.5, prompt='Subtle ill-defined pulmonary opacity')),
    ('small + skip 0.5',  dict(box=0.03, pad=16, skip=0.5)),
    ('all three',         dict(box=0.03, pad=16, skip=0.5,
                               prompt='Subtle ill-defined pulmonary opacity')),
]

for label, kw in COMBOS:
    r, img = run_one(label, **kw)
    sweep_rows.append(r); sweep_imgs[label] = img

sweep = pd.DataFrame(sweep_rows)
print('\n', sweep.to_string(index=False))
sweep.to_csv(IMAGE_DIR / 'hard_case_sweep.csv', index=False)

In [ ]:
n = len(sweep_imgs) + 1
fig, ax = plt.subplots(1, n, figsize=(2.7*n, 3.2))
ax[0].imshow(canvas, cmap='gray'); ax[0].set_title('canvas', fontsize=8); ax[0].axis('off')
for a, (label, img) in zip(ax[1:], sweep_imgs.items()):
    r = sweep[sweep.case == label].iloc[0]
    a.imshow(img, cmap='gray')
    a.set_title(f'{label}\nedit {r.edit_inside} · det {r.det_edited}', fontsize=7)
    a.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
PROMPTS = [
    ('nodule (baseline)',  'Right upper lobe pulmonary nodule'),
    ('small nodule',       'Small pulmonary nodule'),
    ('faint nodule',       'Faint pulmonary nodule'),
    ('subtle nodule',      'Subtle pulmonary nodule'),
    ('tiny nodule',        'Tiny pulmonary nodule'),
    ('ill-defined nodule', 'Small ill-defined pulmonary nodule'),
    ('opacity (control)',  'Subtle ill-defined pulmonary opacity'),
]

prompt_rows, prompt_imgs = [], {}
for label, text in PROMPTS:
    r, img = run_one(label, prompt=text)
    prompt_rows.append(r); prompt_imgs[label] = img

pr = pd.DataFrame(prompt_rows)
print('\n', pr[['case','prompt','edit_inside','det_edited']].to_string(index=False))

In [ ]:
seed_rows = []
for text in ['Small pulmonary nodule', 'Faint pulmonary nodule',
             'Right upper lobe pulmonary nodule']:
    for sd in [0, 1, 42, 123, 7]:
        p = {**BASE, 'prompt': text}
        mask, target, _ = make_mask(p['cx'], p['cy'], p['box'], p['pad'])
        torch.manual_seed(sd)
        out = edit(text, weights=[p['guidance']], image=canvas, edit_mask=mask,
                   keep_mask=ImageOps.invert(mask), num_inference_steps=p['steps'],
                   invert_prompt='', skip_ratio=p['skip'], output_type='pil')[0]
        ins, _ = edit_inside(canvas, out, mask)
        det = best_at(out, target, DET_SIZE)
        seed_rows.append(dict(prompt=text[:34], seed=sd,
                              edit=round(ins,2), det=round(det,3)))
        print(f'{text[:30]:<32} seed {sd:<4} edit {ins:6.2f}  det {det:.3f}')

sd = pd.DataFrame(seed_rows)
print('\n', sd.groupby('prompt').agg(
    edit_mean=('edit','mean'), edit_min=('edit','min'), edit_max=('edit','max'),
    det_mean=('det','mean'), n_worked=('edit', lambda x: (x>6).sum())).round(2).to_string())

In [ ]:
DIAL = [
    ('nodule',        'Right upper lobe pulmonary nodule'),
    ('small nodule',  'Small right upper lobe pulmonary nodule'),
    ('faint nodule',  'Faint right upper lobe pulmonary nodule'),
    ('subtle nodule', 'Subtle right upper lobe pulmonary nodule'),
    ('tiny nodule',   'Tiny right upper lobe pulmonary nodule'),
    ('ill-defined',   'Faint ill-defined right upper lobe pulmonary nodule'),
]

dial_rows, dial_imgs = [], {}
for label, text in DIAL:
    r, img = run_one(label, prompt=text)
    dial_rows.append(r); dial_imgs[label] = img

dial = pd.DataFrame(dial_rows)
print('\n', dial[['case','prompt','edit_inside','det_edited']].to_string(index=False))

In [ ]:
!pip install SimpleITK

In [ ]:
import numpy as np, pandas as pd, cv2, SimpleITK as sitk, glob, shutil
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt, matplotlib.patches as patches

NODE21_MHA = Path('/content/drive/MyDrive/Algoverse/Misc/node21/images')   # <-- set
GRID_SPEC  = Path('/content/drive/MyDrive/01_grid_spec/synthetic_grid_spec.csv')

OUT = Path('/content/out'); (OUT/'backgrounds').mkdir(parents=True, exist_ok=True)
N_BG, POS, SIZE = 20, (0.32, 0.40), 512

# --- fail here instead of silently skipping all 50 -----------------------------
if not NODE21_MHA.exists():
    hits = glob.glob('/content/drive/MyDrive/**/c0221.mha', recursive=True)
    raise FileNotFoundError(f'{NODE21_MHA} does not exist.\n'
                            f'c0221.mha was found at: {hits[:3] or "nowhere -- is Drive mounted?"}')
print(f'{len(list(NODE21_MHA.glob("*.mha")))} .mha files in folder')

def mha_to_canvas(path, size=SIZE):
    a = sitk.GetArrayFromImage(sitk.ReadImage(str(path))).astype(np.float32)
    a = a.squeeze() if a.ndim == 3 else a
    h, w = a.shape; s = min(h, w)
    a = a[(h-s)//2:(h-s)//2+s, (w-s)//2:(w-s)//2+s]        # crop, never squash
    lo, hi = np.percentile(a, [1, 99])
    a = np.clip((a - lo) / (hi - lo + 1e-8), 0, 1)
    a = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)) \
           .apply((a*255).astype(np.uint8)).astype(np.float32) / 255.0
    a = np.clip(cv2.resize(a, (size,size), interpolation=cv2.INTER_AREA), 0, 1)
    return Image.fromarray((a*255).astype(np.uint8)).convert('RGB'), a, (w, h, s)

def in_lung(arr, cx, cy, r=0.06, max_mean=0.45):
    h, w = arr.shape
    m = float(arr[int((cy-r)*h):int((cy+r)*h), int((cx-r)*w):int((cx+r)*w)].mean())
    return m < max_mean, m

names = sorted(pd.read_csv(GRID_SPEC).source_img_name.unique())
usable, rejected, missing, meta = [], [], [], []
for n in names:
    p = NODE21_MHA / n
    if not p.exists():
        missing.append(n); continue
    img, arr, (w0, h0, side) = mha_to_canvas(p)
    ok, m = in_lung(arr, *POS)
    stem = n.replace('.mha','')
    (usable if ok else rejected).append((stem, img, round(m,3)))
    if ok:
        img.save(OUT/'backgrounds'/f'{stem}.png')
        meta.append(dict(background=stem, orig_w=w0, orig_h=h0, crop_side=side,
                         scale=round(SIZE/side,4), lung_mean=round(m,3),
                         cx=POS[0], cy=POS[1], coord_space='fraction'))
    if len(usable) >= N_BG: break

pd.DataFrame(meta).to_csv(OUT/'backgrounds.csv', index=False)
pd.Series(missing).to_csv(OUT/'missing_backgrounds.csv', index=False, header=['name'])

print(f'{len(names)} in spec -> {len(missing)} missing on disk, '
      f'{len(rejected)} not-in-lung, {len(usable)} usable')
assert usable, 'nothing usable -- check NODE21_MHA and the in_lung threshold'
if len(usable) < N_BG:
    print(f'\nNOTE: only {len(usable)} backgrounds, wanted {N_BG}. '
          f'Still runnable -- the spread across chests is what matters, not the count. '
          f'Below ~8 the spread gets noisy.')

k = min(6, len(usable))
fig, ax = plt.subplots(1, k, figsize=(2.6*k, 3), squeeze=False)
for a_, (n, img, m) in zip(ax.ravel(), usable[:k]):
    a_.imshow(img, cmap='gray')
    a_.add_patch(patches.Circle((POS[0]*SIZE, POS[1]*SIZE), 0.06*SIZE,
                                fill=False, edgecolor='lime', lw=2))
    a_.set_title(f'{n}\nmean {m}', fontsize=8); a_.axis('off')
plt.tight_layout(); plt.savefig(OUT/'background_check.png', dpi=110); plt.show()

In [ ]:
from PIL import ImageOps
import torch

(OUT/'synthetic').mkdir(exist_ok=True); (OUT/'masks').mkdir(exist_ok=True)

# Anatomical location is REQUIRED -- 3-word prompts painted nothing, 0/5 seeds.
CFG = dict(box=0.05, pad=24, skip=0.3, guidance=7.5, steps=200,
           prompt='Right upper lobe pulmonary nodule')

rows = []
for bg_name, bg_img, _ in usable:
    mask, target, mask_px = make_mask(POS[0], POS[1], CFG['box'], CFG['pad'])
    assert mask_px >= 47, f'mask {mask_px}px is below the 47px floor'   # assert, never clamp
    torch.manual_seed(42)
    out = edit(CFG['prompt'], weights=[CFG['guidance']], image=bg_img, edit_mask=mask,
               keep_mask=ImageOps.invert(mask), num_inference_steps=CFG['steps'],
               invert_prompt='', skip_ratio=CFG['skip'], output_type='pil')[0]

    ins, _ = edit_inside(bg_img, out, mask)
    det_ed = best_at(out,    target, DET_SIZE)
    det_bg = best_at(bg_img, target, DET_SIZE)

    out.save(OUT/'synthetic'/f'{bg_name}_edited.png')
    mask.save(OUT/'masks'/f'{bg_name}_mask.png')          # save the mask -- F0 went unseen for lack of this
    rows.append(dict(background=bg_name, mask_px=mask_px, edit_inside=round(ins,2),
                     det_edited=round(det_ed,3), det_bg=round(det_bg,3),
                     gain=round(det_ed-det_bg,3), **{k:v for k,v in CFG.items()}))
    print(f'{bg_name:<10} edit={ins:6.2f}  det={det_ed:.3f}  bg={det_bg:.3f}')

multi = pd.DataFrame(rows)
multi.to_csv(OUT/'multi_background.csv', index=False)

# edit_inside is RECORDED, not used as a gate. skip 0.7 scored 9.44 and still detected
# at 0.966, so the threshold measures "something moved", not "a nodule appeared".
print('\n', multi.det_edited.describe().round(3).to_string())
print(f'\nspread: {multi.det_edited.min():.3f} to {multi.det_edited.max():.3f}')
print(f'backgrounds where the detector already fired before editing: '
      f'{(multi.det_bg > 0.05).sum()} of {len(multi)}')

In [ ]:
from google.colab import files
z = shutil.make_archive('/content/pilot_run', 'zip', OUT)
print(f'{Path(z).stat().st_size/1e6:.1f} MB')
files.download(z)

In [ ]:
import numpy as np, pandas as pd, cv2, SimpleITK as sitk, glob
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt, matplotlib.patches as patches

MHA_DIR = Path('/content/drive/MyDrive/Algoverse/Misc/node21/images')
OUT = Path('/content/out'); OUT.mkdir(parents=True, exist_ok=True)

cands = glob.glob('/content/drive/MyDrive/Algoverse/**/*.csv', recursive=True)
ann = [c for c in cands if any(k in Path(c).name.lower()
       for k in ('metadata','annotat','boxes','node21','label'))]
print('candidate annotation files:')
for c in ann[:15]: print('  ', c)

ANN = Path(ann[0]) if ann else None      # <-- set explicitly if the guess is wrong
assert ANN and ANN.exists(), 'no annotation csv found -- set ANN by hand'

raw = pd.read_csv(ANN)
print(f'\n{ANN.name}: {len(raw)} rows')
print(raw.columns.tolist())
print(raw.head(3).to_string())

In [ ]:
COL = dict(name='img_name', x='x', y='y', w='width', h='height', label='label')  # <-- adjust
DET_SIZE = 800

pos = raw[raw[COL['label']] == 1] if COL['label'] in raw else raw
print(f'{len(pos)} annotated nodules across {pos[COL["name"]].nunique()} images')

def load_and_crop(path):
    a = sitk.GetArrayFromImage(sitk.ReadImage(str(path))).astype(np.float32)
    a = a.squeeze() if a.ndim == 3 else a
    H, W = a.shape; s = min(H, W); x0, y0 = (W-s)//2, (H-s)//2
    c = a[y0:y0+s, x0:x0+s]
    lo, hi = np.percentile(c, [1, 99])
    c = np.clip((c - lo) / (hi - lo + 1e-8), 0, 1)
    c = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)) \
           .apply((c*255).astype(np.uint8)).astype(np.float32) / 255.0
    return Image.fromarray((c*255).astype(np.uint8)).convert('RGB'), (W, H, s, x0, y0)

rows, dropped = [], []
for name, g in pos.groupby(COL['name']):
    fn = name if name.endswith('.mha') else f'{name}.mha'
    p = MHA_DIR / fn
    if not p.exists():
        dropped.append((name, 'file missing')); continue
    img, (W, H, s, x0, y0) = load_and_crop(p)

    for _, r in g.iterrows():
        # original pixels -> fractions of the CROPPED square. Resolution-independent
        # from here on, which is the discipline that would have prevented F0.
        fx0 = (r[COL['x']] - x0) / s
        fy0 = (r[COL['y']] - y0) / s
        fx1 = (r[COL['x']] + r[COL['w']] - x0) / s
        fy1 = (r[COL['y']] + r[COL['h']] - y0) / s

        if not (0 <= fx0 < fx1 <= 1 and 0 <= fy0 < fy1 <= 1):
            dropped.append((name, f'box outside crop: {fx0:.2f},{fy0:.2f},{fx1:.2f},{fy1:.2f}'))
            continue                      # drop, never clamp

        target = (fx0, fy0, fx1, fy1)
        score  = best_at(img, target, DET_SIZE)
        rows.append(dict(img_name=name, orig_w=W, orig_h=H, crop_side=s,
                         box_frac_w=round(fx1-fx0, 4), box_frac_h=round(fy1-fy0, 4),
                         box_px_orig=int(r[COL['w']]),
                         det_score=round(score, 4), hit=int(score >= 0.05),
                         fx0=round(fx0,4), fy0=round(fy0,4),
                         fx1=round(fx1,4), fy1=round(fy1,4)))

real = pd.DataFrame(rows)
real.to_csv(OUT/'real_nodule_scores.csv', index=False)
pd.DataFrame(dropped, columns=['img_name','reason']).to_csv(OUT/'real_dropped.csv', index=False)

print(f'\nscored {len(real)} nodules, dropped {len(dropped)}')
if dropped: print('drop reasons:', pd.Series([d[1].split(':')[0] for d in dropped]).value_counts().to_dict())
print(f'\nhits at 0.05: {real.hit.sum()} / {len(real)}  ({100*real.hit.mean():.1f}%)')
print(real.det_score.describe().round(3).to_string())

In [ ]:
k = 6
fig, ax = plt.subplots(1, k, figsize=(2.8*k, 3.2), squeeze=False)
for a_, (_, r) in zip(ax.ravel(), real.sample(min(k, len(real)), random_state=0).iterrows()):
    fn = r.img_name if r.img_name.endswith('.mha') else f'{r.img_name}.mha'
    img, _ = load_and_crop(MHA_DIR / fn)
    a_.imshow(img, cmap='gray', extent=[0,1,1,0])
    a_.add_patch(patches.Rectangle((r.fx0, r.fy0), r.fx1-r.fx0, r.fy1-r.fy0,
                                   fill=False, edgecolor='lime', lw=2))
    a_.set_title(f'{r.img_name}\ndet {r.det_score:.3f}', fontsize=8)
    a_.set_xlim(0,1); a_.set_ylim(1,0); a_.axis('off')
plt.suptitle('green must sit on an actual nodule. If it does not, the transform is wrong.', fontsize=9)
plt.tight_layout(); plt.savefig(OUT/'real_box_check.png', dpi=110); plt.show()

In [ ]:
# find the split. infer_val.py / the failure-label csv both reference it
import glob
for f in glob.glob('/content/drive/MyDrive/Algoverse/**/*.csv', recursive=True):
    n = Path(f).name.lower()
    if any(k in n for k in ('val','split','fold','failure_label')):
        print(f, '->', pd.read_csv(f).shape)

In [ ]:
SW = pd.DataFrame({'seed': range(20), 'det': [
    0.891, 0.669, 0.717, 0.867, 0.796, 0.933, 0.000, 0.152, 0.561, 0.000,
    0.918, 0.919, 0.299, 0.438, 0.923, 0.000, 0.072, 0.937, 0.000, 0.000]})

In [ ]:
real['approx_mm'] = real.box_frac_w * 350
print(real[real.split=='val'].approx_mm.describe().round(1).to_string())
print(f'\nsynthetic lesion: 50.3 mm')
print(f'val real nodules larger than 50mm: {(real[real.split=="val"].approx_mm>50).sum()} of 144')

# does size explain detection among real nodules?
v = real[real.split=='val']
print(f'\nsize vs detection among real: r = {v.approx_mm.corr(v.det_score):+.3f}')

In [ ]:
VAL = Path('/content/drive/MyDrive/Algoverse/Me/feliciano_week_4/real_node21_failure_labels.csv')                       # <-- set from the list above
val_names = set(pd.read_csv(VAL).iloc[:, 0].astype(str).str.replace('.mha','', regex=False))

real['stem'] = real.img_name.str.replace('.mha', '', regex=False)
real['split'] = np.where(real.stem.isin(val_names), 'val', 'train')
print(real.groupby('split').det_score.agg(['count','mean','median','std']).round(3).to_string())
print(real.groupby('split').hit.mean().round(3).to_string())

v = real[real.split == 'val']
print(f'\nheld-out real: n={len(v)}  mean {v.det_score.mean():.3f}  '
      f'median {v.det_score.median():.3f}  found {v.hit.mean():.1%}')
print(f'synthetic:     n={len(SW)}  mean {SW.det.mean():.3f}  '
      f'median {SW.det.median():.3f}  found {(SW.det>=0.05).mean():.1%}')

from scipy.stats import mannwhitneyu
u, p = mannwhitneyu(v.det_score, SW.det)
print(f'\nMann-Whitney p = {p:.4g}')

In [ ]:
print([n for n in dir() if 'edit' in n.lower() or 'pipe' in n.lower()])

In [ ]:
PIPE = edit         # <-- swap in whatever the line above shows

def generate(bg, prompt, mask, skip=0.3, g=7.5, steps=200, seed=42):
    torch.manual_seed(seed)
    return PIPE(prompt, weights=[g], image=bg, edit_mask=mask,
                keep_mask=ImageOps.invert(mask), num_inference_steps=steps,
                invert_prompt='', skip_ratio=skip, output_type='pil')[0]

def edit_inside(bg, ed, mask):
    b = np.asarray(bg.convert('L'), np.float32); e = np.asarray(ed.convert('L'), np.float32)
    return float(np.abs(e-b)[np.asarray(mask, bool)].mean())

def make_mask(cx, cy, box_frac, pad_px, size=512):
    half = box_frac*size/2
    x0, y0 = cx*size-half-pad_px, cy*size-half-pad_px
    x1, y1 = cx*size+half+pad_px, cy*size+half+pad_px
    assert 0 <= x0 < x1 <= size and 0 <= y0 < y1 <= size, 'mask does not fit'
    m = Image.new('L', (size, size), 0)
    ImageDraw.Draw(m).ellipse([x0, y0, x1, y1], fill=255)
    return m, (x0/size, y0/size, x1/size, y1/size), round(x1-x0, 1)

POS = (0.32, 0.40)
OUT = Path('/content/out'); (OUT/'img').mkdir(parents=True, exist_ok=True)

In [ ]:
for n, v in list(globals().items()):
    if not n.startswith('_') and 'Pipeline' in type(v).__name__:
        print(f'{n:<20} {type(v).__name__}')

In [ ]:
rows = []
for i, (bg_name, bg_img, _) in enumerate(usable):
    for label, box, pad in [('small_47px', 0.005, 22), ('large_74px', 0.05, 24)]:
        m, t, mpx = make_mask(*POS, box, pad, 512)
        if mpx < 47:
            m, t, mpx = make_mask(*POS, 0.01, 22, 512)
        g = generate(bg_img, 'Right upper lobe pulmonary nodule', m, seed=i)
        g.save(OUT/'img'/f'{bg_name}_{label}.png')
        rows.append(dict(background=bg_name, seed=i, size=label, mask_px=mpx,
                         approx_mm=round(350*mpx/512, 1),
                         det=round(best_at(g, t), 3),
                         det_bg=round(best_at(bg_img, t), 3),
                         edit_inside=round(edit_inside(bg_img, g, m), 2)))
        print(f'{bg_name:<10} {label:<11} {rows[-1]["approx_mm"]:>5.1f}mm  '
              f'det {rows[-1]["det"]:.3f}  bg {rows[-1]["det_bg"]:.3f}')

MC = pd.DataFrame(MC_rows := rows); MC.to_csv(OUT/'multi_chest_size.csv', index=False)
print('\n', MC.groupby('size').det.agg(['count','mean','median','std']).round(3).to_string())
print(f'\nbetween-chest σ: {MC[MC["size"]=="large_74px"].det.std():.3f}   '
      f'(single-chest seed σ was 0.395)')
print(f'chests firing pre-edit: {(MC.det_bg>0.05).sum()} of {len(MC)}')

In [ ]:
for lab, g in MC.groupby('size'):
    print(f'\n{lab}:  ' + '  '.join(f'{v:.2f}' for v in sorted(g.det)))
    print(f'  below 0.3: {(g.det<0.3).sum()}/20   0.3-0.7: {g.det.between(0.3,0.7).sum()}   '
          f'above 0.7: {(g.det>0.7).sum()}')

print('\nper-chest, both sizes:')
print(MC.pivot(index='background', columns='size', values='det').round(3).to_string())

from scipy.stats import mannwhitneyu
v = real[real.split=='val'].det_score
for lab, g in MC.groupby('size'):
    u, p = mannwhitneyu(g.det, v)
    print(f'{lab} vs real held-out:  p = {p:.4g}')

In [ ]:
hard = ['c0300', 'c1295', 'c1480']
easy = ['c0458', 'c0788', 'c1650']
show = hard + easy

fig, ax = plt.subplots(2, 6, figsize=(17, 6.2))
for j, name in enumerate(show):
    bgp = next(im for n, im, _ in usable if n == name)
    edp = Image.open(OUT/'img'/f'{name}_large_74px.png')
    d = MC[(MC.background == name) & (MC['size'] == 'large_74px')].det.iloc[0]
    for row, im in [(0, bgp), (1, edp)]:
        ax[row, j].imshow(im, cmap='gray')
        ax[row, j].add_patch(patches.Circle((POS[0]*512, POS[1]*512), 74/2,
                             fill=False, edgecolor='lime' if j >= 3 else 'red', lw=2))
        ax[row, j].axis('off')
    ax[0, j].set_title(f'{name}  det {d:.3f}\n{"HARD" if j < 3 else "easy"}', fontsize=9)
plt.suptitle('top: background   bottom: edited.  Is the circle inside the lung on the red ones?',
             fontsize=11)
plt.tight_layout(); plt.savefig(OUT/'hard_vs_easy_chests.png', dpi=140); plt.show()

bgcsv = pd.read_csv(OUT/'backgrounds.csv')
m = MC[MC['size']=='large_74px'].merge(bgcsv, on='background')
print(m[['background','lung_mean','orig_w','orig_h','crop_side','det']]
      .sort_values('det').to_string(index=False))
print(f'\nlung_mean vs det:  r = {m.lung_mean.corr(m.det):+.3f}')

In [ ]:
print(MC.sort_values('det')[['background','size','edit_inside','det']].to_string(index=False))
print('\nedit_inside vs det:  r = %.3f' % MC.edit_inside.corr(MC.det))
print(MC.groupby(MC.det < 0.3).edit_inside.describe().round(2).to_string())

In [ ]:
ok = MC[MC.edit_inside > 30]
print(f'excluding generation failures: n={len(ok)}  mean {ok.det.mean():.3f}  '
      f'median {ok.det.median():.3f}')
from scipy.stats import mannwhitneyu
print('vs real held-out: p = %.4g' % mannwhitneyu(ok.det, real[real.split=='val'].det_score)[1])